# 交叉验证：K-Fold 评估模型稳定性

本 notebook 使用 `predictive_maintenance.csv` 演示如何使用 **Stratified K-Fold 交叉验证** 评估模型稳定性。

主要内容包括：

1. K-Fold 交叉验证的基本原理；
2. 为什么类别不平衡时使用 `StratifiedKFold`；
3. 使用 Pipeline 避免预处理泄露；
4. 对多个模型进行 5-Fold 交叉验证；
5. 使用均值、标准差和箱线图比较模型稳定性；
6. 在平均性能和波动程度之间做模型选择。

## 数据集说明

目标变量：

- `Machine failure`：`1` 表示设备故障，`0` 表示正常。

输入特征：

- `Type`：设备类型；
- `Air temperature`：空气温度；
- `Process temperature`：过程温度；
- `Rotational speed`：转速；
- `Torque`：扭矩；
- `Tool wear`：刀具磨损。

`TWF`、`HDF`、`PWF`、`OSF`、`RNF` 是故障原因/故障模式指示变量，直接使用会造成标签泄露，因此不作为输入特征。

## 1. K-Fold 交叉验证的核心思想

只做一次训练集/测试集拆分时，模型评估结果可能受到随机划分影响。某一次测试集恰好简单，模型分数就会偏高；某一次测试集恰好困难，分数就会偏低。

K-Fold 交叉验证会把数据分成 `K` 份，并重复训练和评估 `K` 次：

- 第 1 次：第 1 份做验证集，其余做训练集；
- 第 2 次：第 2 份做验证集，其余做训练集；
- ...
- 第 K 次：第 K 份做验证集，其余做训练集。

最后得到 K 个评估分数，通过：

- **均值**：估计模型的平均表现；
- **标准差**：估计模型表现受数据拆分影响的程度，也就是稳定性。

本数据集故障样本只占约 3.4%，所以使用 `StratifiedKFold`。它会在每一折中尽量保持“正常/故障”的比例，避免某一折几乎没有故障样本。

## 2. 导入库

In [ ]:
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 3. 读取数据

In [ ]:
csv_path = Path.cwd() / "predictive_maintenance.csv"

if not csv_path.exists():
    csv_path = Path("Python Guidance/day31-45/predictive_maintenance.csv")

df = pd.read_csv(csv_path)
df.head()

## 4. 定义特征和目标变量

In [ ]:
target = "Machine failure"
failure_mode_columns = ["TWF", "HDF", "PWF", "OSF", "RNF"]

numeric_features = [
    "Air temperature",
    "Process temperature",
    "Rotational speed",
    "Torque",
    "Tool wear",
]
categorical_features = ["Type"]

X = df[numeric_features + categorical_features]
y = df[target]

## 5. 检查目标变量分布

交叉验证的每一折都应尽量保留原始故障比例，因此这里先检查类别分布。

In [ ]:
print("数据形状:", df.shape)
print("缺失值数量:", int(df.isna().sum().sum()))
print("\n目标变量分布:")
print(y.value_counts().rename(index={0: "正常", 1: "故障"}))
print(f"\n故障比例: {y.mean():.2%}")

## 6. 定义预处理 Pipeline

这里统一对数值特征做标准化，对 `Type` 做 One-Hot 编码。

树模型本身对数值尺度不敏感，但统一预处理可以让所有模型使用同一套输入，比较更一致。预处理放进 `Pipeline` 后，每一折都会只在训练折上拟合预处理器，再应用到验证折，避免信息泄露。

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ]
)

## 7. 定义要比较的模型

这里选择 4 个代表模型：

- 逻辑回归：线性基线；
- KNN：距离型模型；
- XGBoost：梯度提升树；
- LightGBM：高效梯度提升树。

由于故障类别很少，逻辑回归使用 `class_weight="balanced"`。

In [ ]:
models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("prep", preprocessor),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=1000,
                    random_state=42,
                ),
            ),
        ]
    ),
    "KNN": Pipeline(
        steps=[
            ("prep", preprocessor),
            (
                "model",
                KNeighborsClassifier(n_neighbors=5, weights="distance"),
            ),
        ]
    ),
    "XGBoost": Pipeline(
        steps=[
            ("prep", preprocessor),
            (
                "model",
                xgb.XGBClassifier(
                    n_estimators=200,
                    learning_rate=0.1,
                    max_depth=3,
                    objective="binary:logistic",
                    eval_metric="auc",
                    tree_method="hist",
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
    "LightGBM": Pipeline(
        steps=[
            ("prep", preprocessor),
            (
                "model",
                lgb.LGBMClassifier(
                    n_estimators=200,
                    learning_rate=0.1,
                    num_leaves=15,
                    min_child_samples=20,
                    objective="binary",
                    random_state=42,
                    n_jobs=-1,
                    force_col_wise=True,
                    verbosity=-1,
                ),
            ),
        ]
    ),
}

list(models.keys())

## 8. 运行 Stratified 5-Fold 交叉验证

这里使用 5 折分层交叉验证，并同时计算：

- Accuracy
- Precision
- Recall
- F1
- ROC-AUC
- PR-AUC（Average Precision，更适合类别不平衡场景）

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

records = []

for model_name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    for fold_index, row in pd.DataFrame(scores).iterrows():
        record = {
            "model": model_name,
            "fold": fold_index + 1,
        }
        for metric_name in scoring:
            record[metric_name] = row[f"test_{metric_name}"]
        records.append(record)

cv_results = pd.DataFrame(records)
cv_results

## 9. 查看每折结果和稳定性指标

重点关注：

- `mean_f1`：模型平均 F1；
- `std_f1`：F1 在不同折之间的波动；
- `mean_recall` / `std_recall`：模型找出故障的能力及稳定性；
- `mean_pr_auc` / `std_pr_auc`：类别不平衡下的整体排序能力。

In [ ]:
summary = (
    cv_results.groupby("model")
    .agg(
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        mean_precision=("precision", "mean"),
        std_precision=("precision", "std"),
        mean_recall=("recall", "mean"),
        std_recall=("recall", "std"),
        mean_f1=("f1", "mean"),
        std_f1=("f1", "std"),
        mean_roc_auc=("roc_auc", "mean"),
        std_roc_auc=("roc_auc", "std"),
        mean_pr_auc=("pr_auc", "mean"),
        std_pr_auc=("pr_auc", "std"),
    )
    .sort_values("mean_f1", ascending=False)
)

summary.round(4)

In [ ]:
best_model_name = summary.index[0]
best_mean_f1 = summary.loc[best_model_name, "mean_f1"]
best_std_f1 = summary.loc[best_model_name, "std_f1"]

print("按平均 F1 排序后的最优模型:", best_model_name)
print(f"平均 F1: {best_mean_f1:.4f}")
print(f"F1 标准差: {best_std_f1:.4f}")

## 10. 用箱线图观察每折波动

箱线图可以直观看到：

- 中位数大致在什么水平；
- 分数分布是否集中；
- 是否存在某一折明显很差；
- 哪个模型更稳定。

In [ ]:
metrics_to_plot = ["f1", "recall", "precision", "pr_auc"]
metric_titles = {
    "f1": "F1",
    "recall": "Recall",
    "precision": "Precision",
    "pr_auc": "PR-AUC",
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for metric, ax in zip(metrics_to_plot, axes.flat):
    sns.boxplot(
        data=cv_results,
        x="model",
        y=metric,
        ax=ax,
        hue="model",
        palette="Set2",
        legend=False,
    )
    sns.stripplot(
        data=cv_results,
        x="model",
        y=metric,
        ax=ax,
        color="black",
        alpha=0.55,
        size=5,
    )
    ax.set_title(metric_titles[metric])
    ax.set_xlabel("")
    ax.set_ylabel("Score")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

## 11. 平均表现 vs. 稳定性

选择模型时，不应只看某一个分数最高的折，也不能只看平均值。更合理的方式是同时考虑：

1. 平均 F1 / Recall 是否足够高；
2. 标准差是否足够低；
3. 最差一折是否仍能满足业务要求；
4. 模型复杂度和训练成本是否可接受。

下面把平均 F1 和 F1 标准差放在一起比较。

In [ ]:
stability_view = summary.reset_index()

plt.figure(figsize=(8, 5))
for _, row in stability_view.iterrows():
    plt.errorbar(
        row["std_f1"],
        row["mean_f1"],
        fmt="o",
        markersize=8,
        capsize=4,
        label=row["model"],
    )
    plt.annotate(
        row["model"],
        (row["std_f1"], row["mean_f1"]),
        textcoords="offset points",
        xytext=(8, 6),
    )

plt.xlabel("F1 标准差（越小越稳定）")
plt.ylabel("平均 F1（越大越好）")
plt.title("模型平均表现与稳定性")
plt.grid(True, alpha=0.3)
plt.show()

## 12. 总结

本 notebook 演示了如何使用 Stratified K-Fold 交叉验证评估模型稳定性：

- K-Fold 使用多轮训练/验证，降低单次随机拆分的偶然性；
- `StratifiedKFold` 能在每一折中保持类别比例，适合故障检测这类不平衡数据；
- Pipeline 保证每折中的标准化和 One-Hot 编码只从训练折学习；
- 均值反映平均性能，标准差反映稳定性；
- 类别不平衡任务中，Accuracy 和 ROC-AUC 可能偏乐观，应重点观察 F1、Recall 和 PR-AUC；
- 最终模型选择应同时考虑平均性能、波动程度、最差折表现和业务成本。

## 可以进一步尝试

1. 使用 `RepeatedStratifiedKFold` 重复多轮 K-Fold，进一步降低随机性；
2. 使用 `cross_val_predict` 获取每折的袋外预测，绘制整体混淆矩阵；
3. 在交叉验证中同时搜索模型参数；
4. 对少数类尝试过采样方法，如 SMOTE；
5. 在业务上定义漏报和误报成本，选择总成本最低的模型与阈值。